# Responsible AI Governance Framework for Cloud AI Services

**Problem Statement:** Develop a governance framework that enforces responsible AI deployment in cloud platforms.  
Covers: policy rules, auditing tools, access control, bias monitoring, and compliance reporting.  
Applicable to organizations deploying cloud AI (AWS, Azure, GCP).

---

## Architecture
```
User → API → Model
           ↓
    Governance Layer
    ├── Bias Monitor          [Section 2]
    ├── Anomaly Detector      [Section 3]
    ├── Policy Engine         [Section 4]
    ├── Access Control        [Section 5]
    └── Logger                [Section 6]
           ↓
    Dashboard + Reports       [Section 7]
```

## Datasets Used
| # | Dataset | Purpose |
|---|---------|--------|
| 1 | Cloud Services Audit Logs (simulated) | Anomaly detection & monitoring |
| 2 | Fairness Benchmark Suites | Bias & compliance checks |
| 3 | Regulatory Corpora (GDPR, CCPA) | Policy mapping frameworks |
| 4 | User Interaction Logs | Accountability & traceability |

---
**Table of Contents**
1. [Setup & Data Generation](#section-1)
2. [Bias Monitor](#section-2)
3. [Anomaly Detector](#section-3)
4. [Policy Engine](#section-4)
5. [Access Control](#section-5)
6. [Logger & Audit Trail](#section-6)
7. [Dashboard & Compliance Reports](#section-7)

---
## Section 1 — Setup & Data Generation <a id='section-1'></a>

We simulate all four datasets realistically.  
In production, replace the `generate_*` functions with actual data loaders (S3, BigQuery, Azure Blob, etc.).

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install fairlearn scikit-learn pandas numpy matplotlib seaborn plotly scipy ipywidgets

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
from collections import defaultdict
import random
import json
import re

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Fairlearn (bias)
try:
    from fairlearn.metrics import (
        MetricFrame,
        demographic_parity_difference,
        demographic_parity_ratio,
        equalized_odds_difference,
        false_positive_rate,
        false_negative_rate,
        selection_rate
    )
    FAIRLEARN_AVAILABLE = True
    print("✓ fairlearn loaded")
except ImportError:
    FAIRLEARN_AVAILABLE = False
    print("⚠ fairlearn not installed — bias section will use manual implementations")

np.random.seed(42)
random.seed(42)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11
})

print("✓ All imports complete")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA GENERATORS
# Each mirrors a real-world cloud AI dataset.
# ─────────────────────────────────────────────────────────────────────────────

def generate_fairness_benchmark(n=2000):
    """
    Fairness Benchmark Suite.
    Simulates an AI hiring / loan approval model output with protected attributes.
    Intentional bias: lower approval rates injected for certain demographic groups.
    """
    np.random.seed(42)
    gender    = np.random.choice(['Male', 'Female', 'Non-binary'], n, p=[0.50, 0.45, 0.05])
    race      = np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Other'], n,
                                  p=[0.45, 0.20, 0.15, 0.15, 0.05])
    age_group = np.random.choice(['18-30', '31-45', '46-60', '60+'], n, p=[0.30, 0.35, 0.25, 0.10])

    # Objective features
    credit_score = np.random.normal(680, 80, n).clip(300, 850).astype(int)
    income       = np.random.lognormal(10.8, 0.6, n).astype(int)
    debt_ratio   = np.random.beta(2, 5, n).round(3)
    years_emp    = np.random.poisson(7, n)

    # Ground truth: outcome based on objective features
    score = (credit_score - 300) / 550 * 0.5 + (income / 200000) * 0.3 - debt_ratio * 0.2
    y_true = (score + np.random.normal(0, 0.08, n) > 0.38).astype(int)

    # Model prediction with injected demographic bias
    bias_term = np.zeros(n)
    bias_term[gender == 'Female']     -= 0.07
    bias_term[gender == 'Non-binary'] -= 0.12
    bias_term[race == 'Black']        -= 0.10
    bias_term[race == 'Hispanic']     -= 0.08
    bias_term[age_group == '60+']     -= 0.06

    pred_prob  = (score + bias_term + np.random.normal(0, 0.06, n)).clip(0, 1)
    y_pred     = (pred_prob > 0.40).astype(int)
    y_pred_prob = pred_prob.round(4)

    return pd.DataFrame({
        'sample_id':    [f'S{i:05d}' for i in range(n)],
        'gender':       gender,
        'race':         race,
        'age_group':    age_group,
        'credit_score': credit_score,
        'income':       income,
        'debt_ratio':   debt_ratio,
        'years_emp':    years_emp,
        'y_true':       y_true,
        'y_pred':       y_pred,
        'y_pred_prob':  y_pred_prob
    })


def generate_audit_logs(n=3000):
    """
    Cloud Services Audit Logs (simulated).
    Mimics AWS CloudTrail / Azure Monitor logs with injected anomalous events.
    """
    np.random.seed(42)
    services   = ['S3', 'EC2', 'Lambda', 'IAM', 'SageMaker', 'Bedrock', 'RDS']
    actions    = ['GetObject', 'PutObject', 'DeleteObject', 'InvokeEndpoint',
                  'CreateRole', 'AttachPolicy', 'DescribeInstances', 'RunInstances',
                  'StopInstances', 'InvokeModel', 'CreateBucket', 'DeleteBucket']
    regions    = ['us-east-1', 'us-west-2', 'eu-west-1', 'ap-south-1']
    statuses   = ['Success', 'Success', 'Success', 'Success', 'Failed', 'Denied']

    base_time = datetime(2024, 1, 1)
    timestamps = [base_time + timedelta(minutes=int(x))
                  for x in np.cumsum(np.random.exponential(2.5, n))]

    n_users = 50
    user_ids = [f'user_{i:03d}' for i in range(n_users)]

    # Normal activity
    user_col    = np.random.choice(user_ids, n)
    service_col = np.random.choice(services, n, p=[0.25,0.20,0.15,0.10,0.12,0.10,0.08])
    action_col  = np.random.choice(actions,  n)
    region_col  = np.random.choice(regions,  n, p=[0.50,0.25,0.15,0.10])
    status_col  = np.random.choice(statuses, n)
    latency_ms  = np.random.lognormal(4.5, 0.5, n).round(1)   # ms
    bytes_trans = np.random.lognormal(8, 1.5, n).astype(int)
    api_calls_per_min = np.random.poisson(3, n)

    # Inject anomalies (~5%)
    anomaly_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
    labels = np.zeros(n, dtype=int)
    labels[anomaly_idx] = 1

    # Anomaly types: brute-force, data exfil, privilege escalation, unusual region
    for i in anomaly_idx:
        atype = np.random.choice(['brute_force', 'data_exfil', 'priv_esc', 'unusual_region'])
        if atype == 'brute_force':
            status_col[i] = 'Failed'
            api_calls_per_min[i] = np.random.randint(80, 200)
        elif atype == 'data_exfil':
            bytes_trans[i] = np.random.randint(500_000_000, 2_000_000_000)
            action_col[i]  = 'GetObject'
        elif atype == 'priv_esc':
            action_col[i]  = np.random.choice(['CreateRole', 'AttachPolicy'])
            service_col[i] = 'IAM'
        elif atype == 'unusual_region':
            region_col[i] = np.random.choice(['cn-north-1', 'sa-east-1', 'me-south-1'])

    return pd.DataFrame({
        'log_id':             [f'L{i:06d}' for i in range(n)],
        'timestamp':          timestamps,
        'user_id':            user_col,
        'service':            service_col,
        'action':             action_col,
        'region':             region_col,
        'status':             status_col,
        'latency_ms':         latency_ms,
        'bytes_transferred':  bytes_trans,
        'api_calls_per_min':  api_calls_per_min,
        'is_anomaly':         labels
    })


def generate_regulatory_corpus():
    """
    Regulatory Corpora (GDPR, CCPA) — structured policy rules.
    Each row is a compliance rule with its domain, severity, and enforcement logic.
    """
    rules = [
        # GDPR
        ('GDPR', 'Art.5',  'data_minimisation',        'HIGH',   'Only data necessary for the stated purpose shall be collected.',
         'features_used <= purpose_required_features'),
        ('GDPR', 'Art.6',  'lawful_basis',              'HIGH',   'Personal data processing must have a lawful basis.',
         'consent_obtained == True OR legitimate_interest_documented == True'),
        ('GDPR', 'Art.9',  'special_category_data',     'CRITICAL','Processing special category data requires explicit consent.',
         'special_attributes_used == False OR explicit_consent == True'),
        ('GDPR', 'Art.13', 'transparency',              'MEDIUM', 'Data subjects must be informed about automated decision-making.',
         'explainability_provided == True'),
        ('GDPR', 'Art.17', 'right_to_erasure',          'HIGH',   'Data subjects can request deletion of their data.',
         'deletion_mechanism_available == True'),
        ('GDPR', 'Art.22', 'automated_decision_making', 'HIGH',   'Solely automated decisions with significant impact require human review.',
         'human_review_available == True OR decision_impact == low'),
        ('GDPR', 'Art.25', 'privacy_by_design',         'MEDIUM', 'Privacy protections must be built into system design.',
         'pii_encryption == True AND access_controls_documented == True'),
        ('GDPR', 'Art.32', 'security_measures',         'HIGH',   'Appropriate technical and organisational measures must ensure data security.',
         'encryption_at_rest == True AND audit_logging == True'),
        ('GDPR', 'Art.33', 'breach_notification',       'CRITICAL','Data breaches must be reported to supervisory authority within 72 hours.',
         'breach_detection_sla_hours <= 72'),
        # CCPA
        ('CCPA', 'Sec.1798.100', 'right_to_know',       'MEDIUM', 'Consumers have the right to know what personal data is collected.',
         'data_inventory_available == True'),
        ('CCPA', 'Sec.1798.105', 'right_to_delete',     'HIGH',   'Consumers can request deletion of personal information.',
         'deletion_mechanism_available == True'),
        ('CCPA', 'Sec.1798.110', 'right_to_access',     'MEDIUM', 'Consumers can access their personal information.',
         'data_export_available == True'),
        ('CCPA', 'Sec.1798.120', 'right_to_opt_out',    'HIGH',   'Consumers can opt out of sale of personal information.',
         'opt_out_mechanism == True'),
        ('CCPA', 'Sec.1798.125', 'non_discrimination',  'HIGH',   'Consumers shall not be discriminated against for exercising CCPA rights.',
         'service_parity_for_opt_out == True'),
        ('CCPA', 'Sec.1798.150', 'data_breach_liability','CRITICAL','Businesses are liable for data breaches of non-encrypted data.',
         'encryption_at_rest == True AND encryption_in_transit == True'),
        # AI-specific (emerging)
        ('AI_ACT', 'Art.9',  'risk_management',         'HIGH',   'High-risk AI systems must have a risk management system.',
         'risk_assessment_documented == True'),
        ('AI_ACT', 'Art.10', 'data_governance',         'HIGH',   'Training data must meet quality criteria and be checked for biases.',
         'bias_audit_conducted == True AND data_quality_score >= 0.8'),
        ('AI_ACT', 'Art.13', 'transparency_ai',         'MEDIUM', 'High-risk AI systems must be transparent and provide information to users.',
         'model_card_available == True AND explainability_score >= 0.7'),
        ('AI_ACT', 'Art.14', 'human_oversight',         'HIGH',   'High-risk AI systems must allow effective human oversight.',
         'override_mechanism == True AND audit_logging == True'),
        ('AI_ACT', 'Art.15', 'accuracy_robustness',     'MEDIUM', 'High-risk AI must achieve appropriate levels of accuracy and robustness.',
         'model_accuracy >= 0.85 AND fairness_score >= 0.80'),
    ]
    return pd.DataFrame(rules, columns=[
        'regulation', 'article', 'rule_id', 'severity',
        'description', 'enforcement_condition'
    ])


def generate_user_interaction_logs(n=1500):
    """
    User Interaction Logs — traces every API call made to the AI system.
    Used for accountability, traceability, and access control enforcement.
    """
    np.random.seed(42)
    roles       = ['admin', 'data_scientist', 'auditor', 'developer', 'readonly']
    endpoints   = ['/v1/predict', '/v1/explain', '/v1/batch', '/v1/retrain',
                   '/v1/models', '/v1/audit', '/v1/delete', '/v1/export']
    # Access matrix: which roles can access which endpoints
    allowed_map = {
        '/v1/predict':  ['admin', 'data_scientist', 'developer', 'readonly'],
        '/v1/explain':  ['admin', 'data_scientist', 'auditor', 'developer'],
        '/v1/batch':    ['admin', 'data_scientist', 'developer'],
        '/v1/retrain':  ['admin', 'data_scientist'],
        '/v1/models':   ['admin', 'data_scientist', 'developer'],
        '/v1/audit':    ['admin', 'auditor'],
        '/v1/delete':   ['admin'],
        '/v1/export':   ['admin', 'auditor'],
    }

    base_time = datetime(2024, 1, 1)
    records = []
    for i in range(n):
        role     = np.random.choice(roles, p=[0.10, 0.30, 0.15, 0.35, 0.10])
        endpoint = np.random.choice(endpoints)
        allowed  = role in allowed_map[endpoint]
        # Inject ~8% unauthorized attempts
        if np.random.random() < 0.08:
            endpoint = np.random.choice(['/v1/retrain', '/v1/delete', '/v1/export'])
            allowed  = role in allowed_map[endpoint]
        ts = base_time + timedelta(seconds=int(np.cumsum(np.random.exponential(60, n))[i]))
        records.append({
            'interaction_id': f'I{i:06d}',
            'timestamp':      ts,
            'user_id':        f'user_{np.random.randint(0,50):03d}',
            'role':           role,
            'endpoint':       endpoint,
            'http_method':    np.random.choice(['GET','POST','DELETE'], p=[0.4,0.5,0.1]),
            'response_code':  200 if allowed else np.random.choice([401, 403]),
            'latency_ms':     round(np.random.lognormal(4.8, 0.4), 1),
            'payload_size_kb':round(np.random.lognormal(3, 1), 2),
            'access_granted': allowed,
            'ip_address':     f'{np.random.randint(10,200)}.{np.random.randint(0,255)}.'
                              f'{np.random.randint(0,255)}.{np.random.randint(1,254)}'
        })
    return pd.DataFrame(records)


# ── Generate all datasets ────────────────────────────────────────────────────
df_fairness  = generate_fairness_benchmark(n=2000)
df_audit     = generate_audit_logs(n=3000)
df_policy    = generate_regulatory_corpus()
df_interact  = generate_user_interaction_logs(n=1500)

print("Datasets generated:")
print(f"  Fairness Benchmark  : {df_fairness.shape}")
print(f"  Audit Logs          : {df_audit.shape}")
print(f"  Regulatory Corpus   : {df_policy.shape}")
print(f"  User Interaction    : {df_interact.shape}")

In [ ]:
# Quick preview of each dataset
print("=== Fairness Benchmark (first 3 rows) ===")
display(df_fairness.head(3))
print("\n=== Audit Logs (first 3 rows) ===")
display(df_audit.head(3))
print("\n=== Regulatory Corpus ===")
display(df_policy[['regulation','article','rule_id','severity']].head(10))
print("\n=== User Interaction Logs (first 3 rows) ===")
display(df_interact.head(3))

---
## Section 2 — Bias Monitor <a id='section-2'></a>

**Dataset:** Fairness Benchmark Suites + User Interaction Logs  
**Goal:** Detect and quantify demographic bias in AI model predictions.

### Metrics implemented
| Metric | Definition | Threshold |
|--------|------------|----------|
| Demographic Parity Ratio | P(ŷ=1\|A=a) / P(ŷ=1\|A=b) | ≥ 0.80 |
| Demographic Parity Difference | P(ŷ=1\|A=a) - P(ŷ=1\|A=b) | ≤ 0.10 |
| Equalized Odds Difference | max(|ΔTP|, |ΔFP|) across groups | ≤ 0.10 |
| Disparate Impact | min group rate / max group rate | ≥ 0.80 (EEOC 80% rule) |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1  Exploratory Analysis — Protected Attribute Distributions
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Protected Attribute Distributions — Fairness Benchmark', fontsize=13, fontweight='bold')

palette = {'Male': '#4C78A8', 'Female': '#F58518', 'Non-binary': '#72B7B2'}

for ax, col in zip(axes, ['gender', 'race', 'age_group']):
    counts = df_fairness[col].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=[palette.get(c, '#B279A2') for c in counts.index],
                  edgecolor='white', linewidth=0.8)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(int(bar.get_height())), ha='center', fontsize=9)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print("Class balance (y_true):", df_fairness['y_true'].value_counts().to_dict())
print("Positive prediction rate (y_pred):", f"{df_fairness['y_pred'].mean():.3f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2  Fairness Metric Engine
# ─────────────────────────────────────────────────────────────────────────────

class BiasMonitor:
    """
    Governance component: Bias Monitor.
    Computes demographic fairness metrics across protected attributes.
    Flags violations against configurable thresholds.
    """

    THRESHOLDS = {
        'dp_ratio':   (0.80, 'ge'),   # disparate impact ≥ 0.80
        'dp_diff':    (0.10, 'le'),   # demographic parity difference ≤ 0.10
        'eod_diff':   (0.10, 'le'),   # equalized odds difference ≤ 0.10
        'di_score':   (0.80, 'ge'),   # EEOC 80% rule
    }

    def __init__(self, df, y_true_col='y_true', y_pred_col='y_pred'):
        self.df        = df.copy()
        self.y_true    = df[y_true_col].values
        self.y_pred    = df[y_pred_col].values
        self.results   = {}
        self.violations = []

    # ── Core metric computations ──────────────────────────────────────────────

    def _positive_rate(self, mask):
        return self.y_pred[mask].mean() if mask.sum() > 0 else 0

    def _tpr(self, mask):
        m = mask & (self.y_true == 1)
        return self.y_pred[m].mean() if m.sum() > 0 else 0

    def _fpr(self, mask):
        m = mask & (self.y_true == 0)
        return self.y_pred[m].mean() if m.sum() > 0 else 0

    def compute_group_metrics(self, sensitive_attr):
        groups  = self.df[sensitive_attr].unique()
        metrics = {}
        for g in groups:
            mask = (self.df[sensitive_attr] == g).values
            n_g  = mask.sum()
            pos_rate = self._positive_rate(mask)
            tpr      = self._tpr(mask)
            fpr      = self._fpr(mask)
            acc      = (self.y_pred[mask] == self.y_true[mask]).mean()
            metrics[g] = {
                'n':           n_g,
                'pos_rate':    round(pos_rate, 4),
                'tpr':         round(tpr, 4),
                'fpr':         round(fpr, 4),
                'accuracy':    round(acc, 4)
            }
        return pd.DataFrame(metrics).T

    def compute_fairness_metrics(self, sensitive_attr):
        groups    = self.df[sensitive_attr].unique()
        pos_rates = {g: self._positive_rate((self.df[sensitive_attr]==g).values)
                     for g in groups}
        tprs      = {g: self._tpr((self.df[sensitive_attr]==g).values) for g in groups}
        fprs      = {g: self._fpr((self.df[sensitive_attr]==g).values) for g in groups}

        max_pr = max(pos_rates.values())
        min_pr = min(pos_rates.values())

        dp_ratio   = min_pr / max_pr if max_pr > 0 else 1.0
        dp_diff    = max_pr - min_pr
        di_score   = dp_ratio

        # Equalized odds difference: max(|ΔTPR|, |ΔFPR|)
        tpr_diff = max(tprs.values()) - min(tprs.values())
        fpr_diff = max(fprs.values()) - min(fprs.values())
        eod_diff = max(tpr_diff, fpr_diff)

        result = {
            'attribute':  sensitive_attr,
            'dp_ratio':   round(dp_ratio, 4),
            'dp_diff':    round(dp_diff, 4),
            'eod_diff':   round(eod_diff, 4),
            'di_score':   round(di_score, 4),
            'max_group':  max(pos_rates, key=pos_rates.get),
            'min_group':  min(pos_rates, key=pos_rates.get),
            'max_pr':     round(max_pr, 4),
            'min_pr':     round(min_pr, 4)
        }
        return result

    def _check_threshold(self, value, threshold, direction):
        if direction == 'ge': return value >= threshold
        if direction == 'le': return value <= threshold
        return True

    def run(self, sensitive_attrs=None):
        if sensitive_attrs is None:
            sensitive_attrs = ['gender', 'race', 'age_group']
        self.violations = []

        for attr in sensitive_attrs:
            fm = self.compute_fairness_metrics(attr)
            gm = self.compute_group_metrics(attr)
            fm['group_metrics'] = gm
            self.results[attr] = fm

            for metric, (threshold, direction) in self.THRESHOLDS.items():
                val   = fm[metric]
                passed = self._check_threshold(val, threshold, direction)
                if not passed:
                    self.violations.append({
                        'attribute': attr,
                        'metric':    metric,
                        'value':     val,
                        'threshold': threshold,
                        'direction': direction,
                        'severity':  'HIGH' if metric in ['di_score','eod_diff'] else 'MEDIUM'
                    })
        return self

    def summary(self):
        rows = []
        for attr, fm in self.results.items():
            rows.append({
                'Attribute':  attr,
                'DP Ratio':   fm['dp_ratio'],
                'DP Diff':    fm['dp_diff'],
                'EOD Diff':   fm['eod_diff'],
                'DI Score':   fm['di_score'],
                'Status':     '✗ FAIL' if any(v['attribute']==attr for v in self.violations) else '✓ PASS'
            })
        return pd.DataFrame(rows).set_index('Attribute')


# Run the Bias Monitor
monitor = BiasMonitor(df_fairness)
monitor.run()

print("=== BIAS MONITOR — Fairness Metric Summary ===")
summary_df = monitor.summary()
display(summary_df)

print(f"\nTotal violations detected: {len(monitor.violations)}")
if monitor.violations:
    print(pd.DataFrame(monitor.violations).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.3  Group-Level Positive Rate Breakdown
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Positive Prediction Rate by Protected Group', fontsize=13, fontweight='bold')

attrs = ['gender', 'race', 'age_group']
colors_map = plt.cm.tab10.colors

for ax, attr in zip(axes, attrs):
    gm = monitor.results[attr]['group_metrics']
    gm_sorted = gm.sort_values('pos_rate', ascending=False)

    bars = ax.bar(gm_sorted.index, gm_sorted['pos_rate'].astype(float),
                  color=colors_map[:len(gm_sorted)], edgecolor='white', linewidth=0.8)

    # Add 80% threshold line (EEOC Disparate Impact)
    max_rate = float(gm_sorted['pos_rate'].max())
    threshold_line = max_rate * 0.80
    ax.axhline(threshold_line, color='red', linestyle='--', linewidth=1.2,
               label=f'80% threshold ({threshold_line:.2f})')

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.2f}', ha='center', fontsize=9)

    ax.set_title(attr.replace('_', ' ').title())
    ax.set_ylabel('Positive Rate')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.4  Disparity Heatmap — Pairwise Group Comparison
# ─────────────────────────────────────────────────────────────────────────────

def pairwise_disparity_matrix(df, y_pred_col, sensitive_attr):
    """Compute pairwise Disparate Impact matrix between all groups."""
    groups = sorted(df[sensitive_attr].unique())
    rates  = {g: df[df[sensitive_attr]==g][y_pred_col].mean() for g in groups}
    n      = len(groups)
    mat    = np.ones((n, n))
    for i, ga in enumerate(groups):
        for j, gb in enumerate(groups):
            if rates[gb] > 0:
                mat[i, j] = rates[ga] / rates[gb]
    return pd.DataFrame(mat, index=groups, columns=groups)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Pairwise Disparate Impact Matrix (row / col positive rate)',
             fontsize=13, fontweight='bold')

for ax, attr in zip(axes, ['gender', 'race', 'age_group']):
    mat = pairwise_disparity_matrix(df_fairness, 'y_pred', attr)
    # Highlight cells below 0.80 in red
    cmap = sns.diverging_palette(10, 133, s=80, l=55, as_cmap=True)
    sns.heatmap(mat, ax=ax, cmap=cmap, vmin=0.6, vmax=1.4, center=1.0,
                annot=True, fmt='.2f', linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': 'DI ratio'})
    ax.set_title(attr.replace('_', ' ').title())
    ax.set_xlabel('Reference group')
    ax.set_ylabel('Comparison group')
    ax.tick_params(axis='both', labelsize=9)

plt.tight_layout()
plt.show()
print("Values < 0.80 (warm/red) indicate potential disparate impact violations (EEOC 80% rule).")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.5  TPR / FPR Comparison (Equalized Odds breakdown)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('TPR vs FPR by Group — Equalized Odds Decomposition', fontsize=13, fontweight='bold')

for ax, attr in zip(axes, ['gender', 'race', 'age_group']):
    gm = monitor.results[attr]['group_metrics'].astype(float)
    x  = np.arange(len(gm))
    w  = 0.35
    ax.bar(x - w/2, gm['tpr'], w, label='TPR', color='#4C78A8', edgecolor='white')
    ax.bar(x + w/2, gm['fpr'], w, label='FPR', color='#F58518', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(gm.index, rotation=15, fontsize=9)
    ax.set_title(attr.replace('_', ' ').title())
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.6  Fairlearn MetricFrame (if available) — richer group-level breakdown
# ─────────────────────────────────────────────────────────────────────────────

if FAIRLEARN_AVAILABLE:
    from sklearn.metrics import accuracy_score, precision_score, recall_score

    for attr in ['gender', 'race', 'age_group']:
        mf = MetricFrame(
            metrics={
                'accuracy':   accuracy_score,
                'precision':  precision_score,
                'recall':     recall_score,
                'selection_rate': selection_rate,
            },
            y_true=df_fairness['y_true'],
            y_pred=df_fairness['y_pred'],
            sensitive_features=df_fairness[attr]
        )
        print(f"\n── MetricFrame: {attr} ──")
        display(mf.by_group)
        print(f"  DPD : {demographic_parity_difference(df_fairness['y_true'], df_fairness['y_pred'], sensitive_features=df_fairness[attr]):.4f}")
        print(f"  DPR : {demographic_parity_ratio(df_fairness['y_true'], df_fairness['y_pred'], sensitive_features=df_fairness[attr]):.4f}")
        print(f"  EOD : {equalized_odds_difference(df_fairness['y_true'], df_fairness['y_pred'], sensitive_features=df_fairness[attr]):.4f}")
else:
    print("Install fairlearn for richer MetricFrame output: pip install fairlearn")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.7  Bias Report Card
# ─────────────────────────────────────────────────────────────────────────────

def generate_bias_report(monitor):
    lines = [
        "=" * 65,
        "  BIAS MONITOR — GOVERNANCE REPORT CARD",
        "=" * 65,
        f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"  Total samples evaluated : {len(monitor.df):,}",
        f"  Protected attributes    : {list(monitor.results.keys())}",
        "",
    ]
    for attr, fm in monitor.results.items():
        viols = [v for v in monitor.violations if v['attribute'] == attr]
        status = '✗  FAIL' if viols else '✓  PASS'
        lines.append(f"  Attribute : {attr.upper():15}  [{status}]")
        lines.append(f"    Disparate Impact (80% rule) : {fm['di_score']:.4f}  "
                     f"{'✗ VIOLATION' if fm['di_score'] < 0.8 else '✓ OK'}")
        lines.append(f"    Demographic Parity Ratio    : {fm['dp_ratio']:.4f}")
        lines.append(f"    Demographic Parity Diff     : {fm['dp_diff']:.4f}  "
                     f"{'✗ VIOLATION' if fm['dp_diff'] > 0.10 else '✓ OK'}")
        lines.append(f"    Equalized Odds Diff         : {fm['eod_diff']:.4f}  "
                     f"{'✗ VIOLATION' if fm['eod_diff'] > 0.10 else '✓ OK'}")
        lines.append(f"    Most favoured group         : {fm['max_group']} (rate={fm['max_pr']:.3f})")
        lines.append(f"    Most disadvantaged group    : {fm['min_group']} (rate={fm['min_pr']:.3f})")
        if viols:
            lines.append(f"    ⚠ Violations detected: {len(viols)}")
            lines.append(f"    Recommendation: Apply reweighing or threshold calibration")
            lines.append(f"    for groups: {fm['min_group']}")
        lines.append("")
    lines.append(f"  Total violations : {len(monitor.violations)}")
    lines.append("=" * 65)
    return "\n".join(lines)

report = generate_bias_report(monitor)
print(report)

---
## Section 3 — Anomaly Detector <a id='section-3'></a>

**Dataset:** Cloud Services Audit Logs (simulated)  
**Goal:** Detect suspicious activity — brute force attacks, data exfiltration, privilege escalation.

**Approach:** Ensemble of Isolation Forest (unsupervised) + statistical Z-score flagging.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1  Audit Log EDA
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Cloud Audit Log — Exploratory Analysis', fontsize=13, fontweight='bold')

# Service distribution
svc_counts = df_audit['service'].value_counts()
axes[0,0].bar(svc_counts.index, svc_counts.values,
              color=plt.cm.tab10.colors[:len(svc_counts)], edgecolor='white')
axes[0,0].set_title('API Calls by Service')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=20)

# Latency distribution (normal vs anomaly)
normal_lat   = df_audit[df_audit['is_anomaly']==0]['latency_ms']
anomaly_lat  = df_audit[df_audit['is_anomaly']==1]['latency_ms']
axes[0,1].hist(normal_lat,  bins=50, alpha=0.7, label='Normal',  color='#4C78A8', density=True)
axes[0,1].hist(anomaly_lat, bins=50, alpha=0.7, label='Anomaly', color='#E45756', density=True)
axes[0,1].set_title('Latency Distribution')
axes[0,1].set_xlabel('Latency (ms)')
axes[0,1].legend()

# Bytes transferred (log scale)
axes[1,0].hist(np.log1p(df_audit[df_audit['is_anomaly']==0]['bytes_transferred']),
               bins=50, alpha=0.7, label='Normal',  color='#4C78A8', density=True)
axes[1,0].hist(np.log1p(df_audit[df_audit['is_anomaly']==1]['bytes_transferred']),
               bins=50, alpha=0.7, label='Anomaly', color='#E45756', density=True)
axes[1,0].set_title('Bytes Transferred (log scale)')
axes[1,0].set_xlabel('log(1 + bytes)')
axes[1,0].legend()

# API calls per minute
axes[1,1].hist(df_audit[df_audit['is_anomaly']==0]['api_calls_per_min'],
               bins=30, alpha=0.7, label='Normal',  color='#4C78A8', density=True)
axes[1,1].hist(df_audit[df_audit['is_anomaly']==1]['api_calls_per_min'],
               bins=30, alpha=0.7, label='Anomaly', color='#E45756', density=True)
axes[1,1].set_title('API Calls per Minute')
axes[1,1].set_xlabel('Calls / min')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print(f"Anomaly base rate: {df_audit['is_anomaly'].mean():.3f} ({df_audit['is_anomaly'].sum()} events)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.2  Feature Engineering for Anomaly Detection
# ─────────────────────────────────────────────────────────────────────────────

def engineer_audit_features(df):
    df = df.copy()

    # Encode categoricals
    for col in ['service', 'action', 'region', 'status']:
        df[col + '_enc'] = LabelEncoder().fit_transform(df[col])

    # Log-scale skewed numerics
    df['log_bytes']    = np.log1p(df['bytes_transferred'])
    df['log_latency']  = np.log1p(df['latency_ms'])

    # Failure flag
    df['is_failed']    = (df['status'] == 'Failed').astype(int)
    df['is_denied']    = (df['status'] == 'Denied').astype(int)

    # Unusual region flag (outside main 4)
    main_regions = {'us-east-1', 'us-west-2', 'eu-west-1', 'ap-south-1'}
    df['unusual_region'] = (~df['region'].isin(main_regions)).astype(int)

    # High IAM activity flag
    df['iam_action'] = (df['service'] == 'IAM').astype(int)

    # Per-user call rate (z-score vs all users)
    user_rates = df.groupby('user_id')['api_calls_per_min'].transform('mean')
    df['user_rate_zscore'] = stats.zscore(user_rates.fillna(user_rates.mean()))

    features = ['service_enc', 'action_enc', 'region_enc', 'status_enc',
                'log_bytes', 'log_latency', 'api_calls_per_min',
                'is_failed', 'is_denied', 'unusual_region',
                'iam_action', 'user_rate_zscore']
    return df, features

df_audit_feat, feature_cols = engineer_audit_features(df_audit)
print(f"Feature matrix: {df_audit_feat[feature_cols].shape}")
display(df_audit_feat[feature_cols].describe().round(3))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.3  Isolation Forest Anomaly Detector
# ─────────────────────────────────────────────────────────────────────────────

class AnomalyDetector:
    """
    Governance component: Anomaly Detector.
    Ensemble: Isolation Forest + statistical Z-score + rule-based flags.
    """

    def __init__(self, contamination=0.05):
        self.iforest = IsolationForest(
            n_estimators=200,
            contamination=contamination,
            random_state=42,
            n_jobs=-1
        )
        self.scaler  = StandardScaler()
        self.fitted  = False

    def fit(self, X):
        X_scaled = self.scaler.fit_transform(X)
        self.iforest.fit(X_scaled)
        self.fitted = True
        return self

    def predict(self, X):
        assert self.fitted, "Call fit() first"
        X_scaled = self.scaler.transform(X)
        # IForest: -1 = anomaly, +1 = normal → remap to 1/0
        if_pred  = (self.iforest.predict(X_scaled) == -1).astype(int)
        if_score = -self.iforest.score_samples(X_scaled)  # higher = more anomalous
        return if_pred, if_score

    def rule_based_flags(self, df):
        """Hard rules derived from domain knowledge."""
        flags = pd.Series(0, index=df.index)
        flags |= (df['api_calls_per_min'] > 50).astype(int)               # brute force
        flags |= (df['bytes_transferred'] > 100_000_000).astype(int)      # data exfil
        flags |= (df['status'] == 'Denied').astype(int)                   # access denied
        flags |= (df['unusual_region'] == 1).astype(int)                  # geo anomaly
        flags |= ((df['service']=='IAM') &
                  (df['action'].isin(['CreateRole','AttachPolicy']))).astype(int)  # priv esc
        return flags

    def ensemble_predict(self, X, df_raw):
        """
        Ensemble: flag if IForest OR rule-based trigger.
        Score = weighted combination.
        """
        if_pred, if_score  = self.predict(X)
        rule_flags         = self.rule_based_flags(df_raw).values
        ensemble_pred      = np.maximum(if_pred, rule_flags)
        ensemble_score     = 0.6 * if_score + 0.4 * rule_flags
        return ensemble_pred, ensemble_score.round(4)


X = df_audit_feat[feature_cols].values
y = df_audit['is_anomaly'].values

detector = AnomalyDetector(contamination=0.05)
detector.fit(X)
pred_labels, pred_scores = detector.ensemble_predict(X, df_audit_feat)

df_audit['anomaly_pred']  = pred_labels
df_audit['anomaly_score'] = pred_scores

print("=== Anomaly Detector — Classification Report ===")
print(classification_report(y, pred_labels, target_names=['Normal', 'Anomaly']))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.4  Anomaly Score Distribution & Top Flagged Events
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Anomaly Detector Results', fontsize=13, fontweight='bold')

# Anomaly score distribution
axes[0].hist(df_audit[df_audit['is_anomaly']==0]['anomaly_score'],
             bins=60, alpha=0.7, label='True Normal',  color='#4C78A8', density=True)
axes[0].hist(df_audit[df_audit['is_anomaly']==1]['anomaly_score'],
             bins=30, alpha=0.7, label='True Anomaly', color='#E45756', density=True)
axes[0].set_xlabel('Anomaly Score (higher = more suspicious)')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution')
axes[0].legend()

# Confusion matrix
cm = confusion_matrix(y, pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal','Anomaly'], yticklabels=['Normal','Anomaly'],
            linewidths=0.5)
axes[1].set_title('Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("\n=== Top 10 Highest-Risk Events ===")
top_anomalies = (
    df_audit[df_audit['anomaly_pred']==1]
    .sort_values('anomaly_score', ascending=False)
    [['log_id','timestamp','user_id','service','action',
      'status','bytes_transferred','api_calls_per_min','anomaly_score']]
    .head(10)
)
display(top_anomalies)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.5  Temporal Anomaly Timeline
# ─────────────────────────────────────────────────────────────────────────────

df_audit['hour'] = df_audit['timestamp'].dt.floor('h')
hourly = df_audit.groupby('hour').agg(
    total_events   = ('log_id', 'count'),
    anomaly_events = ('anomaly_pred', 'sum')
).reset_index()
hourly['anomaly_rate'] = hourly['anomaly_events'] / hourly['total_events']

fig, ax1 = plt.subplots(figsize=(15, 4))
ax2 = ax1.twinx()

ax1.fill_between(hourly['hour'], hourly['total_events'],
                 alpha=0.3, color='#4C78A8', label='Total events')
ax2.plot(hourly['hour'], hourly['anomaly_rate'],
         color='#E45756', linewidth=1.2, label='Anomaly rate')
ax2.axhline(0.10, color='darkred', linestyle='--', linewidth=0.8, label='10% threshold')

ax1.set_xlabel('Time')
ax1.set_ylabel('Total events', color='#4C78A8')
ax2.set_ylabel('Anomaly rate', color='#E45756')
ax1.set_title('Anomaly Timeline — Hourly Event Volume + Anomaly Rate', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 4 — Policy Engine <a id='section-4'></a>

**Dataset:** Regulatory Corpora (GDPR, CCPA, AI Act)  
**Goal:** Map compliance rules to system properties, evaluate system state, and flag violations.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1  Policy Engine — Rule Evaluator
# ─────────────────────────────────────────────────────────────────────────────

class PolicyEngine:
    """
    Governance component: Policy Engine.
    Loads regulatory rules and evaluates a system's compliance state.
    Supports GDPR, CCPA, and AI Act rule sets.
    """

    SEVERITY_WEIGHT = {'CRITICAL': 4, 'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}

    def __init__(self, rules_df):
        self.rules = rules_df.copy()

    def evaluate(self, system_state: dict):
        """
        Evaluate system_state dict against all policy rules.
        Returns per-rule pass/fail and overall compliance scores.
        """
        results = []
        for _, rule in self.rules.iterrows():
            passed, reason = self._eval_condition(rule['enforcement_condition'], system_state)
            results.append({
                'regulation':  rule['regulation'],
                'article':     rule['article'],
                'rule_id':     rule['rule_id'],
                'severity':    rule['severity'],
                'description': rule['description'],
                'passed':      passed,
                'reason':      reason
            })
        return pd.DataFrame(results)

    def _eval_condition(self, condition: str, state: dict):
        """
        Simple expression evaluator for policy conditions.
        Supports: ==, !=, >=, <=, >, < and logical AND / OR.
        Substitutes known keys from state; unknown keys → skip (pass).
        """
        try:
            expr = condition
            for key, val in state.items():
                expr = expr.replace(key, repr(val))
            # Replace any unresolved identifiers with True (unknown = assume compliant)
            expr = re.sub(r'\b[a-zA-Z_][a-zA-Z_0-9]*\b', 'True', expr)
            expr = expr.replace('AND', 'and').replace('OR', 'or')
            result = bool(eval(expr))  # noqa: S307 – controlled expression
            return result, "evaluated"
        except Exception as e:
            return True, f"could not evaluate: {e}"

    def compliance_score(self, eval_df):
        """Weighted compliance score 0-100."""
        total  = sum(self.SEVERITY_WEIGHT[s] for s in eval_df['severity'])
        passed = sum(self.SEVERITY_WEIGHT[s]
                     for s, p in zip(eval_df['severity'], eval_df['passed']) if p)
        return round(passed / total * 100, 2) if total > 0 else 100.0

    def report(self, eval_df):
        """Print compliance report by regulation."""
        score = self.compliance_score(eval_df)
        print(f"\n{'='*60}")
        print(f"  POLICY ENGINE — COMPLIANCE REPORT")
        print(f"  Overall Weighted Compliance Score: {score:.1f} / 100")
        print(f"  {'✓ COMPLIANT' if score >= 80 else '✗ NON-COMPLIANT'}")
        print(f"{'='*60}")

        for reg in eval_df['regulation'].unique():
            sub    = eval_df[eval_df['regulation'] == reg]
            n_pass = sub['passed'].sum()
            print(f"\n  {reg}: {n_pass}/{len(sub)} rules passed")
            fails  = sub[~sub['passed']]
            if not fails.empty:
                for _, row in fails.iterrows():
                    print(f"    ✗ [{row['severity']:8}] {row['article']} — {row['rule_id']}")
                    print(f"       {row['description'][:75]}..." if len(row['description'])>75
                          else f"       {row['description']}")


# Define current system state (reflects what your RAI framework implements)
SYSTEM_STATE = {
    'consent_obtained':                    True,
    'legitimate_interest_documented':      True,
    'special_attributes_used':             False,
    'explainability_provided':             True,
    'deletion_mechanism_available':        True,
    'human_review_available':              True,
    'decision_impact':                     'medium',
    'pii_encryption':                      True,
    'access_controls_documented':          True,
    'encryption_at_rest':                  True,
    'audit_logging':                       True,
    'breach_detection_sla_hours':          48,      # hours to detect breach
    'data_inventory_available':            True,
    'data_export_available':               True,
    'opt_out_mechanism':                   True,
    'service_parity_for_opt_out':          True,
    'encryption_in_transit':               True,
    'risk_assessment_documented':          True,
    'bias_audit_conducted':                True,
    'data_quality_score':                  0.87,
    'model_card_available':                False,   # intentional gap
    'explainability_score':                0.65,    # intentional gap
    'override_mechanism':                  True,
    'model_accuracy':                      0.88,
    'fairness_score':                      0.76,    # intentional gap
    'features_used':                       8,
    'purpose_required_features':           10,
}

engine    = PolicyEngine(df_policy)
eval_df   = engine.evaluate(SYSTEM_STATE)
engine.report(eval_df)

print("\n=== Rule-by-Rule Evaluation ===")
display(eval_df[['regulation','article','rule_id','severity','passed']]
        .sort_values(['passed','severity']))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2  Compliance Score Visualization by Regulation
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Policy Engine — Compliance Overview', fontsize=13, fontweight='bold')

# Per-regulation pass/fail bar
reg_summary = eval_df.groupby('regulation').agg(
    passed = ('passed', 'sum'),
    total  = ('passed', 'count')
).reset_index()
reg_summary['failed']  = reg_summary['total'] - reg_summary['passed']
reg_summary['pct']     = reg_summary['passed'] / reg_summary['total'] * 100

x = np.arange(len(reg_summary))
w = 0.35
axes[0].bar(x - w/2, reg_summary['passed'], w, label='Passed', color='#54A24B', edgecolor='white')
axes[0].bar(x + w/2, reg_summary['failed'], w, label='Failed', color='#E45756', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_summary['regulation'])
axes[0].set_title('Rules Passed vs Failed')
axes[0].set_ylabel('Rule Count')
axes[0].legend()

for i, row in reg_summary.iterrows():
    axes[0].text(i, row['passed'] + 0.05, f"{row['pct']:.0f}%", ha='center', fontsize=10, fontweight='bold')

# Severity breakdown of failures
fail_df = eval_df[~eval_df['passed']]
if not fail_df.empty:
    sev_counts = fail_df['severity'].value_counts()
    sev_colors = {'CRITICAL': '#E45756', 'HIGH': '#F58518', 'MEDIUM': '#EECA3B', 'LOW': '#72B7B2'}
    axes[1].bar(sev_counts.index,
                sev_counts.values,
                color=[sev_colors.get(s, 'gray') for s in sev_counts.index],
                edgecolor='white')
    axes[1].set_title('Violations by Severity')
    axes[1].set_ylabel('Count')
    for i, (_, v) in enumerate(sev_counts.items()):
        axes[1].text(i, v + 0.03, str(v), ha='center', fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No violations!', transform=axes[1].transAxes,
                 ha='center', fontsize=14, color='green')

plt.tight_layout()
plt.show()

---
## Section 5 — Access Control <a id='section-5'></a>

**Dataset:** User Interaction Logs  
**Goal:** Enforce Role-Based Access Control (RBAC), detect unauthorized access attempts, and log violations.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.1  RBAC Engine
# ─────────────────────────────────────────────────────────────────────────────

class AccessControlEngine:
    """
    Governance component: Access Control.
    RBAC enforcement with violation logging and risk scoring.
    """

    PERMISSION_MATRIX = {
        'admin':          {'/v1/predict', '/v1/explain', '/v1/batch', '/v1/retrain',
                           '/v1/models',  '/v1/audit',   '/v1/delete', '/v1/export'},
        'data_scientist': {'/v1/predict', '/v1/explain', '/v1/batch', '/v1/retrain', '/v1/models'},
        'auditor':        {'/v1/explain', '/v1/audit', '/v1/export'},
        'developer':      {'/v1/predict', '/v1/explain', '/v1/batch', '/v1/models'},
        'readonly':       {'/v1/predict'},
    }

    # Risk weight per endpoint (higher = more sensitive)
    ENDPOINT_RISK = {
        '/v1/delete':  5,
        '/v1/retrain': 4,
        '/v1/export':  4,
        '/v1/audit':   3,
        '/v1/batch':   2,
        '/v1/models':  2,
        '/v1/explain': 1,
        '/v1/predict': 1,
    }

    def __init__(self):
        self.violations = []
        self.audit_log  = []

    def check_access(self, user_id, role, endpoint):
        allowed = endpoint in self.PERMISSION_MATRIX.get(role, set())
        risk    = self.ENDPOINT_RISK.get(endpoint, 1)
        entry   = {
            'timestamp':  datetime.now().isoformat(),
            'user_id':    user_id,
            'role':       role,
            'endpoint':   endpoint,
            'allowed':    allowed,
            'risk_score': risk if not allowed else 0
        }
        self.audit_log.append(entry)
        if not allowed:
            entry['violation_type'] = 'UNAUTHORIZED_ACCESS'
            self.violations.append(entry)
        return allowed

    def evaluate_log(self, interactions_df):
        """Batch-evaluate all interaction log entries."""
        results = []
        for _, row in interactions_df.iterrows():
            allowed = self.check_access(row['user_id'], row['role'], row['endpoint'])
            results.append(allowed)
        interactions_df = interactions_df.copy()
        interactions_df['rbac_granted'] = results
        interactions_df['access_mismatch'] = (
            interactions_df['access_granted'] != interactions_df['rbac_granted']
        ).astype(int)
        return interactions_df

    def get_violation_summary(self):
        if not self.violations:
            return pd.DataFrame()
        vdf = pd.DataFrame(self.violations)
        return (
            vdf.groupby(['role','endpoint'])
            .agg(attempts=('user_id','count'), risk=('risk_score','first'))
            .reset_index()
            .sort_values('attempts', ascending=False)
        )


ace = AccessControlEngine()
df_interact_rbac = ace.evaluate_log(df_interact)

n_violations = df_interact_rbac['access_mismatch'].sum()
total_denied = (~df_interact_rbac['rbac_granted']).sum()

print(f"Total interactions    : {len(df_interact_rbac):,}")
print(f"RBAC-denied requests  : {total_denied:,} ({total_denied/len(df_interact_rbac)*100:.1f}%)")
print(f"Log/RBAC mismatches   : {n_violations:,}")

print("\n=== Violation Summary by Role × Endpoint ===")
display(ace.get_violation_summary())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.2  Access Control Visualizations
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Access Control Analysis', fontsize=13, fontweight='bold')

# Denied requests by role
denied_by_role = (
    df_interact_rbac[~df_interact_rbac['rbac_granted']]
    .groupby('role').size().sort_values(ascending=False)
)
axes[0].bar(denied_by_role.index, denied_by_role.values,
            color='#E45756', edgecolor='white')
axes[0].set_title('Denied Requests by Role')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

# Endpoint access heatmap (role × endpoint)
heatmap_data = (
    df_interact_rbac.groupby(['role', 'endpoint'])
    .size().unstack(fill_value=0)
)
sns.heatmap(heatmap_data, ax=axes[1], cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.3, cbar_kws={'shrink': 0.8})
axes[1].set_title('Request Volume: Role × Endpoint')
axes[1].tick_params(axis='x', rotation=25)

# Denied rate by endpoint
ep_stats = df_interact_rbac.groupby('endpoint').agg(
    total      = ('rbac_granted', 'count'),
    denied     = ('rbac_granted', lambda x: (~x).sum())
).reset_index()
ep_stats['deny_rate'] = ep_stats['denied'] / ep_stats['total']
ep_stats = ep_stats.sort_values('deny_rate', ascending=False)

colors = ['#E45756' if r > 0.10 else '#4C78A8' for r in ep_stats['deny_rate']]
axes[2].barh(ep_stats['endpoint'], ep_stats['deny_rate'],
             color=colors, edgecolor='white')
axes[2].axvline(0.10, color='darkred', linestyle='--', linewidth=1, label='10% threshold')
axes[2].set_title('Denial Rate by Endpoint')
axes[2].set_xlabel('Denial Rate')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Section 6 — Logger & Audit Trail <a id='section-6'></a>

**Dataset:** User Interaction Logs  
**Goal:** Provide full accountability and traceability for all governance decisions.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.1  Governance Logger
# ─────────────────────────────────────────────────────────────────────────────

class GovernanceLogger:
    """
    Governance component: Logger.
    Central audit trail for all governance events across components.
    Structured entries with timestamp, component, severity, and payload.
    """

    def __init__(self):
        self._log = []

    def log(self, component, event_type, severity, payload: dict):
        self._log.append({
            'timestamp':  datetime.now().isoformat(timespec='milliseconds'),
            'component':  component,
            'event_type': event_type,
            'severity':   severity,
            'payload':    json.dumps(payload)
        })

    def to_df(self):
        return pd.DataFrame(self._log)

    def query(self, component=None, severity=None, event_type=None):
        df = self.to_df()
        if component:  df = df[df['component']  == component]
        if severity:   df = df[df['severity']   == severity]
        if event_type: df = df[df['event_type'] == event_type]
        return df


# ── Populate the logger from all governance results ──────────────────────────

logger = GovernanceLogger()

# Bias Monitor events
for viol in monitor.violations:
    logger.log(
        component='BiasMonitor',
        event_type='BIAS_VIOLATION',
        severity=viol['severity'],
        payload=viol
    )
logger.log('BiasMonitor', 'SCAN_COMPLETE', 'INFO',
           {'samples': len(df_fairness), 'violations': len(monitor.violations)})

# Anomaly Detector events
for _, row in top_anomalies.head(5).iterrows():
    logger.log(
        component='AnomalyDetector',
        event_type='ANOMALY_FLAGGED',
        severity='HIGH',
        payload={'log_id': row['log_id'], 'user': row['user_id'],
                 'score': row['anomaly_score'], 'action': row['action']}
    )
logger.log('AnomalyDetector', 'SCAN_COMPLETE', 'INFO',
           {'total': len(df_audit), 'flagged': int(pred_labels.sum())})

# Policy Engine events
fail_rules = eval_df[~eval_df['passed']]
for _, row in fail_rules.iterrows():
    logger.log(
        component='PolicyEngine',
        event_type='POLICY_VIOLATION',
        severity=row['severity'],
        payload={'regulation': row['regulation'], 'article': row['article'],
                 'rule_id': row['rule_id']}
    )
logger.log('PolicyEngine', 'EVALUATION_COMPLETE', 'INFO',
           {'score': engine.compliance_score(eval_df), 'violations': len(fail_rules)})

# Access Control events
for v in ace.violations[:5]:
    logger.log('AccessControl', 'UNAUTHORIZED_ATTEMPT', 'HIGH',
               {'user': v['user_id'], 'role': v['role'], 'endpoint': v['endpoint']})
logger.log('AccessControl', 'RBAC_EVALUATION_COMPLETE', 'INFO',
           {'total_requests': int(len(df_interact)), 'denied': int(total_denied)})

print("=== Governance Audit Trail ===")
log_df = logger.to_df()
display(log_df.head(20))
print(f"\nTotal log entries: {len(log_df)}")
print("Severity breakdown:")
print(log_df['severity'].value_counts().to_string())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2  Audit Trail Visualization
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Governance Audit Trail Overview', fontsize=13, fontweight='bold')

# Events by component
comp_counts = log_df['component'].value_counts()
axes[0].bar(comp_counts.index, comp_counts.values,
            color=['#4C78A8','#F58518','#54A24B','#B279A2'], edgecolor='white')
axes[0].set_title('Log Entries by Component')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=10)

# Severity distribution
sev_order  = ['CRITICAL', 'HIGH', 'MEDIUM', 'INFO', 'LOW']
sev_colors_map = {'CRITICAL': '#E45756', 'HIGH': '#F58518',
                  'MEDIUM': '#EECA3B', 'INFO': '#4C78A8', 'LOW': '#72B7B2'}
sev_cnt = log_df['severity'].value_counts().reindex(
    [s for s in sev_order if s in log_df['severity'].values])
axes[1].bar(sev_cnt.index, sev_cnt.values,
            color=[sev_colors_map[s] for s in sev_cnt.index], edgecolor='white')
axes[1].set_title('Log Entries by Severity')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## Section 7 — Dashboard & Compliance Reports <a id='section-7'></a>

**Goal:** Aggregate all governance results into a final compliance report and a dashboard-ready data export.

This section produces:
1. A printable **Executive Compliance Report**
2. **Dashboard summary plots** (replicates what `app.py` would render in Streamlit)
3. **JSON export** for downstream systems

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.1  Governance Score Aggregator
# ─────────────────────────────────────────────────────────────────────────────

def compute_governance_scores():
    """Aggregate component-level scores into a single governance dashboard."""

    # ── Bias Score: fraction of attributes passing all fairness thresholds
    n_attrs   = len(monitor.results)
    n_passing = sum(1 for attr in monitor.results
                    if not any(v['attribute']==attr for v in monitor.violations))
    bias_score = round(n_passing / n_attrs * 100, 1)

    # ── Anomaly Score: 1 – anomaly_rate (capped)
    anom_rate   = pred_labels.mean()
    anom_score  = round(max(0, (1 - anom_rate * 5)) * 100, 1)

    # ── Policy Score: weighted compliance score from Policy Engine
    policy_score = engine.compliance_score(eval_df)

    # ── Access Control Score: 1 – denial_rate
    denial_rate = total_denied / len(df_interact)
    ac_score    = round(max(0, 1 - denial_rate * 2) * 100, 1)

    # ── Overall: weighted average
    overall = round(0.30*bias_score + 0.25*anom_score + 0.30*policy_score + 0.15*ac_score, 1)

    return {
        'Bias Monitor':        bias_score,
        'Anomaly Detector':    anom_score,
        'Policy Engine':       policy_score,
        'Access Control':      ac_score,
        'Overall Governance':  overall
    }


gov_scores = compute_governance_scores()

print("=" * 50)
print("  RAI GOVERNANCE SCORES")
print("=" * 50)
for component, score in gov_scores.items():
    bar_len = int(score / 5)
    status  = '✓' if score >= 80 else '✗'
    print(f"  {status} {component:22} {score:5.1f}/100  {'█'*bar_len}")
print("=" * 50)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.2  Master Dashboard Visualization
# ─────────────────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 14))
fig.suptitle('RAI Governance Framework — Master Dashboard', fontsize=15, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35)

# ── A: Governance Score Gauges ────────────────────────────────────────────────
ax_scores = fig.add_subplot(gs[0, :2])
components = list(gov_scores.keys())[:-1]  # exclude Overall
scores     = [gov_scores[c] for c in components]
colors_bar = ['#54A24B' if s >= 80 else '#E45756' for s in scores]

bars = ax_scores.barh(components, scores, color=colors_bar, edgecolor='white', height=0.5)
ax_scores.axvline(80, color='black', linestyle='--', linewidth=1, label='80% threshold')
ax_scores.set_xlim(0, 105)
ax_scores.set_title('Component Governance Scores', fontweight='bold')
ax_scores.set_xlabel('Score (0-100)')
for bar, score in zip(bars, scores):
    ax_scores.text(score + 1, bar.get_y() + bar.get_height()/2,
                   f'{score:.1f}', va='center', fontsize=10)
ax_scores.legend(fontsize=9)

# ── B: Overall Gauge (donut) ──────────────────────────────────────────────────
ax_gauge = fig.add_subplot(gs[0, 2])
ov  = gov_scores['Overall Governance']
col = '#54A24B' if ov >= 80 else '#E45756'
ax_gauge.pie([ov, 100-ov], colors=[col, '#e0e0e0'],
             startangle=90, counterclock=False,
             wedgeprops={'width': 0.45, 'edgecolor': 'white'})
ax_gauge.text(0, 0, f'{ov}\n/100', ha='center', va='center',
              fontsize=14, fontweight='bold')
ax_gauge.set_title('Overall Score', fontweight='bold')

# ── C: Violation Summary ──────────────────────────────────────────────────────
ax_viols = fig.add_subplot(gs[0, 3])
viol_data = {
    'Bias\nViolations':      len(monitor.violations),
    'Anomalies\nFlagged':    int(pred_labels.sum()),
    'Policy\nFailures':      len(eval_df[~eval_df['passed']]),
    'Unauthorized\nAccess':  total_denied,
}
ax_viols.bar(viol_data.keys(), viol_data.values(),
             color=['#E45756','#F58518','#EECA3B','#B279A2'], edgecolor='white')
ax_viols.set_title('Issues by Category', fontweight='bold')
ax_viols.set_ylabel('Count')

# ── D: Bias — DI Scores ───────────────────────────────────────────────────────
ax_bias = fig.add_subplot(gs[1, :2])
for attr in ['gender', 'race', 'age_group']:
    gm       = monitor.results[attr]['group_metrics'].astype(float)
    x_labels = [f"{attr[:3]}:{g}" for g in gm.index]
    ax_bias.bar(x_labels, gm['pos_rate'].values, alpha=0.75,
                label=attr, edgecolor='white')
ax_bias.axhline(0, color='black', linewidth=0.5)
ax_bias.set_title('Positive Rate by Demographic Group', fontweight='bold')
ax_bias.set_ylabel('Rate')
ax_bias.legend(fontsize=9)
ax_bias.tick_params(axis='x', rotation=45, labelsize=7)

# ── E: Anomaly Score Time Series ──────────────────────────────────────────────
ax_anom = fig.add_subplot(gs[1, 2:])
ax_anom.plot(hourly['hour'], hourly['anomaly_rate'],
             color='#E45756', linewidth=1.2, label='Anomaly rate')
ax_anom.fill_between(hourly['hour'], hourly['anomaly_rate'],
                     alpha=0.2, color='#E45756')
ax_anom.axhline(0.10, color='darkred', linestyle='--', linewidth=0.8, label='10% threshold')
ax_anom.set_title('Anomaly Rate Over Time', fontweight='bold')
ax_anom.set_xlabel('Time')
ax_anom.set_ylabel('Rate')
ax_anom.legend(fontsize=9)

# ── F: Policy Compliance Breakdown ────────────────────────────────────────────
ax_pol = fig.add_subplot(gs[2, :2])
pivot  = eval_df.groupby(['regulation','passed']).size().unstack(fill_value=0)
pivot.columns = ['Failed', 'Passed'] if False in pivot.columns else ['Passed']
# Rebuild cleanly
reg_pass = eval_df.groupby('regulation')['passed'].sum()
reg_fail = eval_df.groupby('regulation')['passed'].apply(lambda x: (~x).sum())
x3 = np.arange(len(reg_pass))
ax_pol.bar(x3 - 0.2, reg_pass.values, 0.4, label='Passed', color='#54A24B', edgecolor='white')
ax_pol.bar(x3 + 0.2, reg_fail.values, 0.4, label='Failed', color='#E45756', edgecolor='white')
ax_pol.set_xticks(x3)
ax_pol.set_xticklabels(reg_pass.index)
ax_pol.set_title('Policy Rule Compliance', fontweight='bold')
ax_pol.set_ylabel('Rules')
ax_pol.legend(fontsize=9)

# ── G: Access Control Endpoint Denial ─────────────────────────────────────────
ax_ac = fig.add_subplot(gs[2, 2:])
ep_stats2 = df_interact_rbac.groupby('endpoint').apply(
    lambda x: (~x['rbac_granted']).sum() / len(x)
).sort_values(ascending=False)
colors3 = ['#E45756' if r > 0.10 else '#4C78A8' for r in ep_stats2.values]
ax_ac.barh(ep_stats2.index, ep_stats2.values, color=colors3, edgecolor='white')
ax_ac.axvline(0.10, color='darkred', linestyle='--', linewidth=0.8)
ax_ac.set_title('Denial Rate by Endpoint', fontweight='bold')
ax_ac.set_xlabel('Denial Rate')

plt.savefig('/home/claude/rai_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved to rai_dashboard.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.3  Executive Compliance Report
# ─────────────────────────────────────────────────────────────────────────────

def executive_report():
    bias_viols   = len(monitor.violations)
    anom_flagged = int(pred_labels.sum())
    pol_fails    = len(eval_df[~eval_df['passed']])
    ov_score     = gov_scores['Overall Governance']
    status       = 'COMPLIANT' if ov_score >= 80 else 'NON-COMPLIANT'

    print("\n" + "═"*68)
    print("  RESPONSIBLE AI GOVERNANCE FRAMEWORK")
    print("  EXECUTIVE COMPLIANCE REPORT")
    print("═"*68)
    print(f"  Report Date    : {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"  Platform Scope : AWS / Azure / GCP Cloud AI Services")
    print(f"  Overall Status : {'✓  ' if status=='COMPLIANT' else '✗  '}{status}")
    print(f"  Governance Score: {ov_score:.1f} / 100")
    print("═"*68)

    print("\n  COMPONENT SCORES")
    print("  " + "-"*44)
    for comp, score in gov_scores.items():
        if comp == 'Overall Governance': continue
        s = '✓' if score >= 80 else '✗'
        print(f"  {s} {comp:22} {score:5.1f}/100")

    print("\n  FINDINGS SUMMARY")
    print("  " + "-"*44)
    print(f"  Bias violations detected      : {bias_viols}")
    print(f"  Most disadvantaged (gender)   : {monitor.results['gender']['min_group']}")
    print(f"  Most disadvantaged (race)     : {monitor.results['race']['min_group']}")
    print(f"  Anomalous events flagged      : {anom_flagged} / {len(df_audit)} ({anom_flagged/len(df_audit)*100:.1f}%)")
    print(f"  Policy rules failed           : {pol_fails} / {len(df_policy)}")
    print(f"  Unauthorized access attempts  : {total_denied} / {len(df_interact)}")

    print("\n  REGULATORY COMPLIANCE")
    print("  " + "-"*44)
    for reg in eval_df['regulation'].unique():
        sub  = eval_df[eval_df['regulation'] == reg]
        pct  = sub['passed'].mean() * 100
        icon = '✓' if pct == 100 else '✗'
        print(f"  {icon} {reg:10}  {pct:.0f}% rules compliant")

    print("\n  KEY RECOMMENDATIONS")
    print("  " + "-"*44)
    if bias_viols > 0:
        print("  1. Apply post-processing threshold calibration to reduce")
        print(f"     bias against: {monitor.results['gender']['min_group']}, {monitor.results['race']['min_group']}")
    if pol_fails > 0:
        for _, row in eval_df[~eval_df['passed']].iterrows():
            print(f"  2. Remediate [{row['regulation']} {row['article']}] — {row['rule_id']}")
    if total_denied > len(df_interact) * 0.05:
        print("  3. Review RBAC permissions — high unauthorized access rate.")
    print("  4. Enable continuous bias monitoring pipeline (daily scans).")
    print("  5. Establish human review workflow for high-risk predictions.")
    print("\n" + "═"*68)

executive_report()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.4  JSON Export (for downstream systems / Streamlit dashboard)
# ─────────────────────────────────────────────────────────────────────────────

export = {
    'report_timestamp':  datetime.now().isoformat(),
    'governance_scores': gov_scores,
    'bias_monitor': {
        'violations':     len(monitor.violations),
        'by_attribute':   {
            attr: {
                'di_score': fm['di_score'],
                'dp_ratio': fm['dp_ratio'],
                'dp_diff':  fm['dp_diff'],
                'eod_diff': fm['eod_diff'],
            } for attr, fm in monitor.results.items()
        }
    },
    'anomaly_detector': {
        'total_events':    len(df_audit),
        'flagged':         int(pred_labels.sum()),
        'anomaly_rate':    round(pred_labels.mean(), 4)
    },
    'policy_engine': {
        'compliance_score':  engine.compliance_score(eval_df),
        'rules_failed':      len(eval_df[~eval_df['passed']]),
        'failed_rules':      eval_df[~eval_df['passed']][['regulation','article','rule_id','severity']]
                             .to_dict('records')
    },
    'access_control': {
        'total_requests':    len(df_interact),
        'denied_requests':   total_denied,
        'denial_rate':       round(total_denied / len(df_interact), 4)
    },
    'audit_log_summary': {
        'total_entries':     len(log_df),
        'by_severity':       log_df['severity'].value_counts().to_dict()
    }
}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if hasattr(obj, 'item'): return obj.item()  # numpy scalars
        if hasattr(obj, 'tolist'): return obj.tolist()
        return super().default(obj)

with open('/home/claude/governance_report.json', 'w') as f:
    json.dump(export, f, indent=2, cls=NumpyEncoder)

print("JSON report exported to governance_report.json")
print(json.dumps(export, indent=2))

---
## Next Steps — Streamlit Dashboard

To launch the interactive dashboard, create `app.py` with the following structure and run `streamlit run app.py`:

```python
# app.py — RAI Governance Dashboard (Streamlit)
import streamlit as st
import json, pandas as pd

st.set_page_config(page_title='RAI Governance', layout='wide')
st.title('Responsible AI Governance Dashboard')

# Load JSON export from notebook
with open('governance_report.json') as f:
    report = json.load(f)

# Sidebar — component selector
page = st.sidebar.radio('Component', 
    ['Overview', 'Bias Monitor', 'Anomaly Detector', 'Policy Engine', 'Access Control'])

if page == 'Overview':
    cols = st.columns(5)
    for col, (name, score) in zip(cols, report['governance_scores'].items()):
        col.metric(name, f"{score}/100", delta=None)

elif page == 'Bias Monitor':
    threshold = st.sidebar.slider('DI Threshold', 0.70, 0.95, 0.80)
    st.subheader('Fairness Metrics by Attribute')
    # Load and re-run BiasMonitor with selected threshold ...

# (Extend similarly for other pages)
```

---
*End of RAI Governance Framework Notebook*  
*All governance components — Bias Monitor, Anomaly Detector, Policy Engine, Access Control, Logger — are implemented and interconnected above.*